In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from plot_serializer.matplotlib.serializer import MatplotlibSerializer

os.chdir("../../../")

from src.postprocessing.utils import find_absolute_min_max, find_strategy_arg_min_max
from src.postprocessing.utils_plots import create_standard_list, sort_list_of_dicts, merge_standard_list, add_legend_right, process_dicts, extract_h5_data_fan_acoustics, save_data_raw

plt.style.use("FST.mplstyle")

from matplotlib.font_manager import FontProperties
arial_font = FontProperties(family="Arial", style="italic")

outdirectory = "plots/GPZ/"

In [ ]:
general_information = {"author": {"name": "Julius H.P. Breuer", "ORCID": "0000-0002-6226-7208"}, "type": "PhD thesis, Technische Universität Darmstadt, Chair of Fluid Systems", "status": "in preparation", "title": "Algorithmische Systemplanung raumlufttechnischer Anlagen", "figure_created_with": "analyse_GPZ.ipynb"}

def save_data(fig, serializer, outname, filedata, caption):
    save_data_raw(fig, serializer, outdirectory, outname, filedata, caption, general_information=general_information, id_result_subfolder = "id_results/")

In [ ]:
pt = 1./72.27
diss_tex_width = 390.0*pt

In [ ]:
# CONTROL STRATEGY NAMES:
CAV = "KVS"
VAVCPC = "KONST/ZENTRAL"
VAVVPC = "VAR/ZENTRAL"
DFCPC = "KONST/VERTEILT"
ONLYDF = "OHNE/VERTEILT"
ODSCC = "OPT"

control_strategies_names_ger = [CAV, VAVCPC, VAVVPC, DFCPC, ONLYDF, ODSCC]

control_strategies_names_ger_twocolumn = ["KVS", "KONST/\nZENTRAL", "VAR/\nZENTRAL", "KONST/\nVERTEILT", "OHNE/\nVERTEILT", "OPT"]

control_strategies_names_opt = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

markerstyle = ["o","D", "s","v","^","P"]

In [ ]:
out_directory = "plots/diss/"

save_plots = input("Save all plots? y/n")
save_plots = True if save_plots.lower() == "y" else False

## Comparison to real building

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        "results/GPZ/combined/real/duct_dimensions_limited_by_real/VAV-VPC/",
        "results/GPZ/combined/real/real_fans_var_ducts_within_bounds/VAV-VPC/",
        "results/GPZ/combined/real/var_fans_fixed_ducts/VAV-VPC/",
        "results/GPZ/combined/real/real_fans_fixed_ducts/VAV-VPC/",
        ]

    # "FIXED FANS, VAR",

    names = ["OPT", "KANAL\nDIM.", "VENTILATOR\nAUSWAHL", "REAL"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=False)

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    for curr_dict in standard_dict_list:
        # Compute exact fan energy costs
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring

    for idx, curr_dict in enumerate(standard_dict_list):
        costs = [curr_dict[name][0]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    ax.set_ylabel("LEBENSZYKLUSKOSTEN\nin $10^3~$€")

    ax.set_xticks(names, names, fontsize=9) #multialignment="left", 
    ax = add_legend_right(ax, entry_names)


    ax.axhline(sum(costs),color="black")

    exact_lcc = np.array([x["exact_lcc"][0] for x in standard_dict_list])
    lcc_savings = (1-exact_lcc/np.max(exact_lcc))*100
    for i in range(3):
        ax.text(i,sum(costs)+1,f"${lcc_savings[i]:.0f}~\%$",horizontalalignment="center", fontsize=9)
        ax.annotate(
            '',
            xy=(i, sum(costs)*1.008),      # top end
            xytext=(i, exact_lcc[i]/1e3*0.988),  # bottom end
            arrowprops=dict(
                arrowstyle=f'<|-|>,head_length={0.5/(i+1)},head_width={0.1/(i+1)}', # {0.1/(i+1)}
                lw=1,
                mutation_scale=18,
                shrinkA=0,
                shrinkB=0,
                color="black",),
        )

    real_data_topo = standard_dict_list[-1]

    fig.tight_layout()

    if save_plots:
        outname = "bar_chart_compare_to_real_GPZ"
        filedata = [x["filename"][0] for x in standard_dict_list]
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Vergleich der auf Lebenszykluskosten optimierten Lösung mit der realen Lösung, sowie optimalen Teillösungen, wie die Kanaldimensionierung und die Ventilatorauswahl. Im Vergleich zur realen Lösung lassen sich \SI{16}{\percent} Lebenszykluskosten durch die optimierte Topologie einsparen. Die Optimierung allein der Kanaldimensionen führt zu einer Reduktion um \SI{10}{\percent}, die optimale Ventilatorauswahl spart hingegen nur \SI{6}{\percent} ein."
        save_data(fig, serializer, outname, filedata, caption)

## LCC Bar charts

In [ ]:
standard_case_bar_chart = True

if standard_case_bar_chart:
    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    standard_dict_list

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Compute indices of maximum LCC
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring


    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        # Extract relevant costs at arg_max_lcc index
        costs = [curr_dict[name][arg_max_lcc[idx]]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    ax.set_xticks(ticknames, ticknames, fontsize=7.5, ) #multialignment="left"
    
    ax = add_legend_right(ax, entry_names)

    ax.set_ylim(top=9e1)

    # ax.set_xticks(foldernames, foldernames, rotation=90)

    fig.tight_layout()
    
    if save_plots:
        filedata = [x["filename"][0] if len(x["filename"]) == 1 else 0 for x in standard_dict_list]
        outname = "bar_chart_topo_GPZ"
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Vergleich der Lösungen minimaler Lebenszykluskosten für verschiedene Regelstrategien. \gls{vavvpc} ist die optimale Strategie hinsichtlich Lebenszykluskosten -- aber auch Energieverbrauch. Der Energieverbrauch ist gegenüber \gls{cav} um \SI{85}{\percent} reduziert. Die Kanaldimensionierung unterscheidet sich zwischen den Regelstrategien nur geringfügig."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_pareto = True
if standard_case_pareto:

    power_consumption = "exact_power_consumption"

    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    foldernames = control_strategies_names_opt[2:]
    markerstyle_curr = markerstyle[2:]

    filename_list = [folder + x + "/" for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    serializer = MatplotlibSerializer()
    # fig,ax = serializer.subplots(figsize=(9,4))
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    for idx, curr_dict in enumerate(standard_dict_list):
        ax.plot(curr_dict[power_consumption],curr_dict["invest_costs"]/1e3, "-", color="k", markeredgecolor="k")
        ax.plot(curr_dict[power_consumption],curr_dict["invest_costs"]/1e3, color="k", marker=markerstyle_curr[idx],markerfacecolor="white", markeredgecolor="k", label=control_strategies_names_ger[2:][idx])
        min_lcc_arg = np.argmin(curr_dict["exact_lcc"])
        ax.plot([curr_dict[power_consumption][min_lcc_arg]], [curr_dict["invest_costs"][min_lcc_arg]/1e3], markerstyle_curr[idx], markeredgecolor="k", markerfacecolor="k")

        min_energy_arg = np.argmin(curr_dict[power_consumption])
        min_invest_arg = np.argmin(curr_dict["invest_costs"])
        # if idx == 0 or idx == len(standard_dict_list)-1:
        ax.plot([curr_dict[power_consumption][-1]]*2, [curr_dict["invest_costs"][-1]/1e3, 1e2], zorder=0, color="black")
        ax.plot([curr_dict[power_consumption][0],1e3], [curr_dict["invest_costs"][0]/1e3]*2, zorder=0, color="black")

    ax.plot(real_data_topo[power_consumption], real_data_topo["invest_costs"]/1e3, marker="s", markeredgecolor="k", markerfacecolor="black")
    plt.annotate(
        "REAL",
        xy=(real_data_topo[power_consumption], real_data_topo["invest_costs"]/1e3),              # point being annotated
        xytext=(620, 52),      # where the text goes
        arrowprops=dict(arrowstyle="-", color="grey"),
        fontsize=8,
    )

    min_power, max_power = find_absolute_min_max(standard_dict_list, power_consumption)

    min_lcc, max_lcc = find_absolute_min_max(standard_dict_list, "exact_lcc")

    min_invest, max_invest = find_absolute_min_max(standard_dict_list, "invest_costs")

    ax.set_ylabel("KAPITALBEZOGENE\nKOSTEN in $10^3~$€")
    ax.set_xlabel("MITTLERE LEISTUNG in W")

    x = np.array([min_power*0.9,max_power*1.05])
    power_to_cost = standard_dict_list[0]["fan_energy_costs"][0]/1e3 / standard_dict_list[0]["power_consumption"][0]
    power_to_cost = power_to_cost
    for i in np.linspace(min_lcc/1e3, max_lcc/1e3, 5):
        # plt.plot(x,i - 5*power_to_cost*x, "--",color="grey",zorder=-1)
        plt.plot(x,i - power_to_cost*x, "--",color="grey",zorder=-1)

    ax.fill_between(curr_dict[power_consumption], curr_dict["invest_costs"]/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)
    ax.fill_between([0,curr_dict[power_consumption][-1]], max_invest * 1.05/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)
    ax.fill_between([curr_dict[power_consumption][0],max_power*1.05], curr_dict["invest_costs"][0]/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)

    ax.set_xlim(min_power*0.97, max_power*1.05)
    ax.set_ylim(min_invest*0.97/1e3, max_invest*1.02/1e3)
    # ax.set_ylim(top=80)

    isoline = ax.text(500, 7.32e1, "Isolinie: Lebenszykluskosten", fontsize=8, rotation=-3.5, fontfamily='Arial', fontstyle='italic',) #, backgroundcolor="white"
    # isoline.set_bbox(dict(facecolor='white', alpha=0.7, edgecolor="white"))

    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)

    # for x in standard_dict_list[-3]["exact_power_consumption"]:
    #     ax.axvline(x, color="blue")
    # ax.axvline(490, color="red")

    fig.tight_layout()
    
    if save_plots:
        outname = "pareto_lcc_GPZ_zoom"
        filedata = [y for x in standard_dict_list for y in x["filename"]]
    
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Pareto-Front der konfliktären Ziele Leistungsaufnahme und kapitalbezogene Kosten aller Regelstrategien. Die ausgefüllten Marker entsprechen dem optimalen Kompromiss für Laufzeit $T=\SI{12}{a}$. Die real verbaute Lösung ist eingezeichnet. Der optimale Kompromiss wird für alle Regelstrategien nahe der minimalen kapitalbezogenen Kosten erreicht -- die Leistungsaufnahme spielt demnach bei allen Regelstrategien eine untergeordnete Rolle. Regelstrategie \gls{cav} und \gls{vavcpc} liegen außerhalb des betrachteten Ausschnitts. Die gestrichelten Linien zeigen Isolinien der Lebenszykluskosten."
        save_data(fig, serializer, outname, filedata, caption)

#### Acoustic solution (even though acoustics is supposed to come later)

In [ ]:
[*control_strategies_names_opt[2:4], *control_strategies_names_opt[5:]]

In [ ]:
standard_case_pareto = True
if standard_case_pareto:

    power_consumption = "exact_power_consumption"

    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    markerstyle_curr = [*markerstyle[2:4], *markerstyle[5:]]

    foldernames = [*control_strategies_names_opt[2:4], *control_strategies_names_opt[5:]]

    filename_list = [folder + x + "/" for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Configuration", use_connected_opt=True)

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring

    serializer = MatplotlibSerializer()
    # fig,ax = serializer.subplots(figsize=(9,4))
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    for idx, curr_dict in enumerate(standard_dict_list):
        ax.plot(curr_dict[power_consumption],curr_dict["invest_costs"]/1e3, "-", color="k", markeredgecolor="k")
        ax.plot(curr_dict[power_consumption],curr_dict["invest_costs"]/1e3, color="k", marker=markerstyle_curr[idx],markerfacecolor="white", markeredgecolor="k", label=control_strategies_names_ger[2:][idx])
        min_lcc_arg = np.argmin(curr_dict["exact_lcc"])
        ax.plot([curr_dict[power_consumption][min_lcc_arg]], [curr_dict["invest_costs"][min_lcc_arg]/1e3], markerstyle_curr[idx], markeredgecolor="k", markerfacecolor="k")

        ax.plot([curr_dict[power_consumption][-1]]*2, [curr_dict["invest_costs"][-1]/1e3, 1e2], zorder=0, color="black")
        ax.plot([curr_dict[power_consumption][0],1e3], [curr_dict["invest_costs"][0]/1e3]*2, zorder=0, color="black")

    real_data_config = {power_consumption: 648.41, "invest_costs": (39152+25909)}

    ax.plot(real_data_config[power_consumption], real_data_config["invest_costs"]/1e3, marker="s", markeredgecolor="k", markerfacecolor="black")
    plt.annotate(
        "REAL",
        xy=(real_data_config[power_consumption], real_data_config["invest_costs"]/1e3),              # point being annotated
        xytext=(620, 60),      # where the text goes
        arrowprops=dict(arrowstyle="-", color="grey"),
        fontsize=8,
    )

    ax.set_ylabel("KAPITALBEZOGENE\nKOSTEN in $10^3~$€")
    ax.set_xlabel("MITTLERE LEISTUNG in W")

    x = np.array([min_power*0.9,max_power*1.05])
    power_to_cost = standard_dict_list[0]["fan_energy_costs"][0]/1e3 / standard_dict_list[0]["power_consumption"][0]
    power_to_cost = power_to_cost
    for i in np.linspace(min_lcc/1e3, max_lcc/1e3, 5):
    #     plt.plot(x,i - 2*power_to_cost*x, "--",color="grey",zorder=-1)
        plt.plot(x,i - power_to_cost*x, "--",color="grey",zorder=-1)

    ax.fill_between(curr_dict[power_consumption], curr_dict["invest_costs"]/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)
    ax.fill_between([0,curr_dict[power_consumption][-1]], max_invest * 1.05/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)
    ax.fill_between([curr_dict[power_consumption][0],max_power*1.05], curr_dict["invest_costs"][0]/1e3, min_invest*0.9/1e3, color="lightgrey",zorder=-2)

    ax.set_xlim(min_power*0.97, max_power*1.05)
    ax.set_ylim(min_invest*0.97/1e3, max_invest*1.02/1e3)
    # ax.set_ylim(top=80)

    isoline = ax.text(520, 7.3e1, "Isogerade: Lebenszykluskosten", fontsize=8, rotation=-3, backgroundcolor="white", fontfamily='Arial', fontstyle='italic')
    isoline.set_bbox(dict(facecolor='white', alpha=0.7, edgecolor="white"))

    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)


    # ax.set_ylim(50,55)
    # ax.set_xlim(500,550)

    fig.tight_layout()
    if save_plots:
        outname = "pareto_lcc_GPZ_config_zoom"
        filedata = [y for x in standard_dict_list for y in x["filename"]]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Pareto-Front der konfliktären Ziele Leistungsaufnahme und kapitalbezogene Kosten aller Regelstrategien. Die ausgefüllten Marker entsprechen dem optimalen Kompromiss für Laufzeit $T=\SI{12}{a}$. Die real verbaute Lösung ist eingezeichnet. Der optimale Kompromiss wird für alle Regelstrategien nahe der minimalen kapitalbezogenen Kosten erreicht -- die Leistungsaufnahme spielt demnach bei allen Regelstrategien eine untergeordnete Rolle. Regelstrategie \gls{cav} und \gls{vavcpc} liegen außerhalb des betrachteten Ausschnitts."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True

if standard_case_bar_chart:
    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    foldernames = ["ODS-CC"]
    # foldernames = ["VAV-VPC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    standard_dict_list = sort_list_of_dicts(standard_dict_list, "exact_power_consumption")[0]

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    standard_dict_list["exact_fan_energy_costs"] = (
        standard_dict_list["fan_energy_costs"]
        * standard_dict_list["exact_power_consumption"]
        / standard_dict_list["power_consumption"]
    )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(standard_dict_list[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring

    n_elements = len(standard_dict_list["exact_lcc"])
    
    # Iterate over strategies
    for idx in range(n_elements):

        # Extract relevant costs at arg_max_lcc index
        costs = [standard_dict_list[name][idx]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(f"{costs[-1]:.2f}", cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(f"{costs[-1]:.2f}", cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    # ax.set_xticks(ticknames, ticknames, fontsize=7.5, multialignment="left")

    ax = add_legend_right(ax, entry_names)

    # ax.set_ylim(top=9e1)

    ax.set_xticks([f"{costs:.2f}" for costs in standard_dict_list["exact_fan_energy_costs"]/1e3], [f"{costs:.1f}" for costs in standard_dict_list["exact_fan_energy_costs"]/1e3], fontsize=7)

    # ax.set_xticks(np.arange(n_elements), [""]*n_elements)
    ax.set_xlabel("ENERGIEKOSTEN in $10^3~$€")

    fig.tight_layout()

    

    if save_plots:
        outname = "bar_chart_pareto_lcc_GPZ"
        filedata = [str(x) for x in standard_dict_list["filename"]]
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Vergleich der Kostenaufteilung für verschiedene Kompromisse zwischen Leistungsaufnahme und kapitalbezogenen Kosten für Regelstrategie \gls{odscc}. Für niedrigere Leistungsaufnahme steigt die Anzahl gekaufter Ventilatoren sowie des Kanalnetzes. VSR werden oberhalb von \SI{600}{\watt} weiterhin vor jeem Raum verbaut, erst darunter werden VSR nicht mehr vor jedem Raum benötigt."
        save_data(fig, serializer, outname, filedata, caption)

# NOISE

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        "results/GPZ/combined/real/duct_dimensions_limited_by_real/VAV-VPC/",
        "results/GPZ/combined/real/real_topo_and_config/VAV-VPC/",
        # "results/GPZ/combined/real/var_fans_fixed_ducts/VAV-VPC/",
        # "results/GPZ/combined/real/real_fans_fixed_ducts/VAV-VPC/",
        ]

    # "FIXED FANS, VAR",

    names = ["OPT","REAL"]

    standard_dict_list = create_standard_list(folders, "Configuration", use_connected_opt=True)

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(0.8*diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    for curr_dict in standard_dict_list:
        # Compute exact fan energy costs
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring

    for idx, curr_dict in enumerate(standard_dict_list):
        costs = [curr_dict[name][0]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    ax.set_xticks(names, names, fontsize=9) #multialignment="left", 
    ax = add_legend_right(ax, entry_names)


    ax.axhline(sum(costs),color="black")

    exact_lcc = np.array([x["exact_lcc"][0] for x in standard_dict_list])
    lcc_savings = (1-exact_lcc/np.max(exact_lcc))*100
    for i in range(len(lcc_savings)-1):
        ax.text(i,sum(costs)+1,f"${lcc_savings[i]:.0f}~\%$",horizontalalignment="center", fontsize=9)
        ax.annotate(
            '',
            xy=(i, sum(costs)*1.005),      # top end
            xytext=(i, exact_lcc[i]/1e3*0.99),  # bottom end
            arrowprops=dict(
                arrowstyle=f'<|-|>,head_length={0.5/(i+1)},head_width={0.1/(i+1)}', # {0.1/(i+1)}
                lw=1,
                mutation_scale=18,
                shrinkA=0,
                shrinkB=0,
                color="black",),
        )

    real_data = standard_dict_list[-1]

    fig.tight_layout()
    
    if save_plots:
        outname = "bar_chart_compare_to_real_GPZ_config"
        filedata = [x["filename"][0] for x in standard_dict_list]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Vergleich der auf Lebenszykluskosten optimierten Lösung mit der realen Lösung. Im Vergleich zur realen Lösung lassen sich insgesamt \SI{20}{\percent} Lebenszykluskosten und \SI{17}{\percent} der elektrischen Leistung durch die kombinierte Optimierung der Topologie und Konfiguration einsparen."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True


if standard_case_bar_chart:
    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    # folder = "new_solutions/off/combined/max_velocity_5_10ms_max_height_04m/capex_reduction/"
    # 
    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    ticknames = control_strategies_names_ger_twocolumn

    standard_dict_list = create_standard_list(filename_list, "Configuration", use_connected_opt=True)

    standard_dict_list

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Compute indices of maximum LCC
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring


    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        # Extract relevant costs at arg_max_lcc index
        costs = [curr_dict[name][arg_max_lcc[idx]]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.set_xticks(ticknames, ticknames, fontsize=7.5)

    ax = add_legend_right(ax, entry_names)


    ax.set_ylim(top=9e1)
    fig.tight_layout()

    
    if save_plots:
        outname = "bar_chart_config_GPZ"
        filedata = [x["filename"][0] if x["filename"][0] else "no file"  for x in standard_dict_list]
    
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Vergleich der Ergebnisse minimaler Lebenszykluskosten für verschiedene Regelstrategien. Durch die Integration der Akustik in den Planungsprozess steigen die Lebenszykluskosten gegenüber \cref{fig:GPZ_bar_chart_topo} nur geringfügig. Für Regelstrategie \gls{onlydf} gibt es keine akustisch zulässige Lösung."
        save_data(fig, serializer, outname, filedata, caption)

## varying noise limits

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/VAV-VPC"
    # folderpath2 = "new_solutions/GPZ/combined/real/standard_case/VAV-VPC"
    folderpath2 = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/ODS-CC"

    standard_dict_list = create_standard_list([folderpath, folderpath2], "Configuration", use_connected_opt=True)
    # curr_dict = merge_standard_list(standard_dict_list)

    standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")

    cs_names = [control_strategies_names_ger[2], control_strategies_names_ger[-1]]

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["DUCT INVEST", "EQ INVEST", "SILENCER INVEST", "VFC INVEST", "FAN INVEST", "FAN ENERGY"]

    for idx, curr_dict in enumerate(standard_dict_list):
        # Compute exact fan energy costs
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]/1e3
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )


        markerstyle_curr = [markerstyle[2], markerstyle[-1]]

        # ax.fill_between(curr_dict["max_noise_value"], curr_dict["invest_costs"],color="lightgrey",zorder=-2)
        # ax.fill_between([0,curr_dict["exact_power_consumption"][0]], curr_dict["invest_costs"][0],color="lightgrey",zorder=-2)
        # ax.fill_between([curr_dict["exact_power_consumption"][-1],1e9], curr_dict["invest_costs"][-1],color="lightgrey",zorder=-2)

        # for idx, curr_dict in enumerate(standard_dict_list):

        ax.plot(curr_dict["max_noise_value"],curr_dict["exact_lcc"]/1e3, "-", color="k", markeredgecolor="k", zorder=-0.5)
        ax.plot(curr_dict["max_noise_value"],curr_dict["exact_lcc"]/1e3, color="k", marker=markerstyle_curr[idx],markersize=5,markerfacecolor="white", markeredgecolor="k", label=cs_names[idx])
        # min_lcc = np.argmin(curr_dict["exact_lcc"])
        # ax.plot([curr_dict["max_noise_value"][min_lcc]], [curr_dict["exact_lcc"][min_lcc]], markerstyle_curr[idx], markersize=7,markeredgecolor="k", markerfacecolor="k")
        ax.plot([curr_dict["max_noise_value"][0]]*2, [curr_dict["exact_lcc"][0]/1e3, 8.7e1], color="black", zorder=-1)

        if idx == len(standard_dict_list) -1 :
            ax.fill_between(curr_dict["max_noise_value"], 6e1, curr_dict["exact_lcc"]/1e3,color="lightgrey",zorder=-2)
            ax.fill_between([24,curr_dict["max_noise_value"][0]], 6e1, 8.7e1,color="lightgrey",zorder=-2)
            ax.fill_between([curr_dict["max_noise_value"][-1],56], 6e1, curr_dict["exact_lcc"][-1]/1e3,color="lightgrey",zorder=-2)


            ax.plot([curr_dict["max_noise_value"][-1], 56],[curr_dict["exact_lcc"][-1]/1e3]*2,  color="black", zorder=-1)

            ax.plot([55, 56],[curr_dict["exact_lcc"][-1]/1e3-1,curr_dict["exact_lcc"][-1]/1e3],  color="black", zorder=-1)
            ax.plot([54.5, 55.5],[curr_dict["exact_lcc"][-1]/1e3-1,curr_dict["exact_lcc"][-1]/1e3],  color="black", zorder=-1)
            ax.plot([54, 55],[curr_dict["exact_lcc"][-1]/1e3-1,curr_dict["exact_lcc"][-1]/1e3],  color="black", zorder=-1)

            ax.plot([24.5,25],[86,87],  color="black", zorder=-1)
            ax.plot([24.5,25],[85.5,86.5],  color="black", zorder=-1)
            ax.plot([24.5,25],[85,86],  color="black", zorder=-1)

            behag = ax.text(25.7, 85.5,'max. akustische\nBehaglichkeit',  ha="left", multialignment="left", backgroundcolor="white", fontfamily='Arial', fontstyle='italic')
            behag.set_bbox(dict(facecolor='white', alpha=0.9, edgecolor="white"))

            laerm = ax.text(53, 65,'min. Lebens-\nzykluskosten',  ha="left", multialignment="left", backgroundcolor="white", fontfamily='Arial', fontstyle='italic')
    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.legend()
    ax.set_xlabel("MAX. ZULÄSSIGER SCHALLDRUCKPEGEL in dB")
    ax.set_ylim((6e1,8.7e1))
    ax.set_xlim((24,56))

    fig.tight_layout()
    if save_plots:
        outname = "pareto_noise"
        filedata = [y for x in standard_dict_list for y in x["filename"]]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Pareto-Front der konfliktären Ziele akustischer Komfort und Lebenszykluskosten, dargestellt für die Regelstrategien \gls{vavvpc} und \gls{odscc}. Nur für hohe akustischen Behaglichkeit (niedrige zulässige Schalldruckpegel) ergeben sich unterschiedliche Konfigurationen. Bei einem zulässigen Schalldruckpegel von \SI{30}{\decibel} sind die Lebenszykluskosten von \gls{odscc} gegenüber \gls{vavvpc} um \SI{12}{\percent} niedriger. Mit verteilten Ventilatoren kann der zulässige Schalldruckpegel sogar auf \SI{25}{\decibel} reduziert werden."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
1- standard_dict_list[-1]["exact_power_consumption"][2] / standard_dict_list[0]["exact_power_consumption"][0]

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/VAV-VPC"
    folderpath2 = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/ODS-CC"

    standard_dict_list_c = create_standard_list([folderpath], "Configuration")
    standard_dict_list_c = sort_list_of_dicts(standard_dict_list_c, "max_noise_value")[0]

    standard_dict_list_d = create_standard_list([folderpath2], "Configuration")
    standard_dict_list_d = sort_list_of_dicts(standard_dict_list_d, "max_noise_value")[0]

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator", "Energiekosten"]

    entry_names = entry_names[1:]
    cost_label = cost_label[1:]
    colors = colors[1:]

    # Compute exact fan energy costs
    standard_dict_list_c["exact_fan_energy_costs"] = (
        standard_dict_list_c["fan_energy_costs"]
        * standard_dict_list_c["exact_power_consumption"]
        / standard_dict_list_c["power_consumption"]
    )

    standard_dict_list_d["exact_fan_energy_costs"] = (
        standard_dict_list_d["fan_energy_costs"]
        * standard_dict_list_d["exact_power_consumption"]
        / standard_dict_list_d["power_consumption"]
    )

    cutoff_last_values = 3


    standard_dict_list_d = {key: [value for idx, value in enumerate(values) if idx < len(values) - cutoff_last_values] for key, values in standard_dict_list_d.items()}

    n_entries_d = len(standard_dict_list_d[entry_names[0]])

    # Plot stacked bar

    t_bar = 0.225

    filenames_c = []

    for idx in range(n_entries_d):

        current_noise_level = np.round(standard_dict_list_d["max_noise_value"][idx],1)
        idx_c = np.where(abs(standard_dict_list_c["max_noise_value"]-current_noise_level)<1e-2)[0]
        if  len(idx_c) == 0:
            costs_c = [0]*len(entry_names)
        else:
            idx_c = idx_c[0]
            costs_c = [standard_dict_list_c[name][idx_c]/1e3 for name in entry_names]
            filenames_c.append(standard_dict_list_c["filename"][idx_c])

        
        costs_d = [standard_dict_list_d[name][idx]/1e3 for name in entry_names]

        # Plot stacked bar
        b_cost_c = 0
        b_cost_d = 0
        for i, (cost_c, cost_d) in enumerate(zip(costs_c, costs_d)):
            if i == len(entry_names) - 1:
                ax.bar(idx-t_bar, cost_c, bottom=b_cost_c, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3, width = t_bar*2)
                ax.bar(idx+t_bar, cost_d, bottom=b_cost_d, color="white", hatch="xx", edgecolor="black", linewidth=0.3, width = t_bar*2)
            else:
                ax.bar(idx-t_bar, cost_c, bottom=b_cost_c, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3, width = t_bar*2)
                ax.bar(idx+t_bar, cost_d, bottom=b_cost_d, color=colors[i], edgecolor="black", linewidth=0.3, width = t_bar*2)
            b_cost_c += cost_c
            b_cost_d += cost_d

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.set_xlabel("MAX. SCHALLDRUCKPEGEL in dB")

    x_ticks = [str(f"{x:.1f}") for x in standard_dict_list_c["max_noise_value"]]
    # ax.set_xticks([str(x) for x in standard_dict_list_c["max_noise_value"]], x_ticks, rotation=90)
    
    ax = add_legend_right(ax, cost_label)

    annotate_VAVVPC = standard_dict_list_c["exact_lcc"][2] - standard_dict_list_c["duct_costs"][2]
    annotate_ODSCC = standard_dict_list_c["exact_lcc"][2] - standard_dict_list_c["duct_costs"][2]

    ax.annotate(
        "VAR/ZENTRAL",
        xy=(4-t_bar, annotate_VAVVPC/1e3),              # point being annotated
        xytext=(4,  50),      # where the text goes
        arrowprops=dict(arrowstyle="-", color="black"),
        ha="right",
        fontsize=8,
    )

    ax.annotate(
        "OPT",
        xy=(4+t_bar, annotate_ODSCC/1e3),              # point being annotated
        xytext=(4.5,  50),      # where the text goes
        arrowprops=dict(arrowstyle="-", color="black"),
        fontsize=8,
    )

    ax.set_xticks(range(n_entries_d),[f"{x:.1f}" for x in standard_dict_list_d["max_noise_value"]], fontsize=9)

    # ax.set_xlim(left=-0.25)

    fig.tight_layout()
    
    if save_plots:
        outname = "GPZ_VAR_VVS_OPT_bar_chart_noise_pareto"
        filedata = [str(x) for x in filenames_c] + [str(x) for x in standard_dict_list_d["filename"]]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Aufschlüsselung der Lebenszykluskosten der unterschiedlichen maximalen Schalldruckpegel für die Regelstrategien \gls{vavvpc} (jeweils linker Balken) und \gls{odscc} (jeweils rechter Balken). Beide Regelstrategien führen für maximale Schalldruckpegel oberhalb von \SI{30}{\decibel} zu identischen Lösungen, somit ist \gls{vavvpc} die optimale Regelstrategie. Für höhere akustische Behaglichkeit steigen bis dahin die Lebenszykluskosten für \gls{vavvpc} primär durch die kapitalbezogenen Kosten der Schalldämpfer. Für \SI{30}{\decibel} steigen bei \gls{vavvpc} die kapitalbezogenen Kosten für Schalldämpfer und die Energiekosten stark an. Für \gls{odscc} steigen die Lebenszykluskosten weniger stark, da verteilte Ventilatoren zum Einsatz kommen und sich somit die optimale Regelstrategie ändert."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/ODS-CC"

    # folderpath = "results/GPZ/combined/real/rooms_identical_noise_limits/ODS-CC"
    # folderpath2 = "results/GPZ/combined/real/standard_case/VAV-VPC"

    standard_dict_list = create_standard_list([folderpath], "Configuration")

    standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator", "Energiekosten"]

    # Compute exact fan energy costs
    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )
    # for idx in range(len(curr_dict["exact_fan_energy_costs"])):
    #     costs = [curr_dict[name][idx] for name in entry_names]

        # Plot stacked bar
    for idx in range(len(curr_dict["exact_fan_energy_costs"])):
        costs = [curr_dict[name][idx]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(str(curr_dict["max_noise_value"][idx]), cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(str(curr_dict["max_noise_value"][idx]), cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.set_xlabel("MAX. ZULÄSSIGER SCHALLDRUCKPEGEL in dB")

    x_ticks = [str(f"{x:.1f}") for x in curr_dict["max_noise_value"]]
    ax.set_xticks([str(x) for x in curr_dict["max_noise_value"]], x_ticks, rotation=90)

    ax = add_legend_right(ax, cost_label)


    fig.tight_layout()
    if save_plots:
        outname = "GPZ_OPT_bar_chart_noise_pareto"
        filedata = [str(x) for x in curr_dict["filename"]]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Aufschlüsselung der Lebenszykluskosten der unterschiedlichen maximalen Schalldruckpegel für die Regelstrategie \gls{odscc}. Für höhere akustische Behaglichkeit steigen die Lebenszykluskosten primär durch die kapitalbezogenen Kosten der Schalldämpfer. Bei der Konfiguration mit höchster akustischer Behaglichkeit wird ein Schalldämpfer mit hohem Druckverlust verbaut, wodurch die Energiekosten stark ansteigen."
        save_data(fig, serializer, outname, filedata, caption)

## weitere Planungsbetrachtungen

#### maximale Kanalhöhe

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        "results/GPZ/combined/duct_variations/max_velocity_5_10ms_max_height_0.5m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_5_10ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_5_10ms_max_height_0.3m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_5_10ms_max_height_0.2m/VAV-VPC/",
        ]

    names = ["0.5","0.4", "0.3", "0.2"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)
    

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    
    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    # Compute exact fan energy costs
    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )
    for idx in range(len(curr_dict["exact_fan_energy_costs"])):
        costs = [curr_dict[name][idx]/1e3 for name in entry_names]

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
   
    ax.set_xlabel("MAXIMALE KANALHÖHE in m")
    ax = add_legend_right(ax, entry_names)

    fig.tight_layout()
    if save_plots:
        outname = "bar_chart_max_height"
        filedata = [str(x) if x else "no file" for x in curr_dict["filename"]]
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Vergleich der Lebenszykluskosten-optimalen Lösungen für verschiedene maximale Kanalhöhen. Die Lebenszykluskosten steigen für niedrigere Kanalhöhen nur geringfügig an. Für eine maximale Kanalhöhe von \SI{0.2}{\meter} hat das Optimierungsproblem keine Lösung mehr."
        save_data(fig, serializer, outname, filedata, caption)

#### maximale Kanalgeschwindigkeit

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        #"results/GPZ/combined/max_height_04m_no_velocity_limit/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_5_5ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_8_8ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_10_10ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_12_12ms_max_height_0.4m/VAV-VPC/",
        ]

    # "FIXED FANS, VAR",

    names = ["5","8","10","12"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    
    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    # Compute exact fan energy costs
    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring

    
    for idx in range(len(curr_dict["exact_fan_energy_costs"])):
        costs = [curr_dict[name][idx]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in €")
    ax.set_xlabel("MAX. KANALGESCHWINDIGKEIT in m/s")
    # ax.legend(cost_label)
    # _ = ax.set_xticks(names, names, rotation=90)

    ax = add_legend_right(ax, entry_names)

    ax.axhline(curr_dict["exact_lcc"][0]/1e3, color="black")

    lcc_savings = (1-curr_dict["exact_lcc"]/curr_dict["exact_lcc"][0])*100

    for i in range(1,4):
        ax.text(i,curr_dict["exact_lcc"][0]/1e3+1,f"${lcc_savings[i]:.0f}~\%$",horizontalalignment="center")
        ax.annotate(
            '',
            xy=(i, curr_dict["exact_lcc"][0]/1e3*1.01),      # top end
            xytext=(i, curr_dict["exact_lcc"][i]/1e3*0.988),  # bottom end
            arrowprops=dict(
                arrowstyle=f'<|-|>,head_length={0.2},head_width={0.05}', # {0.1/(i+1)}
                lw=1,
                mutation_scale=18,
                shrinkA=0,
                shrinkB=0,
                color="black",),
        )

    fig.tight_layout()
    if save_plots:
        outname = "bar_chart_duct_velocity"
        filedata = [str(x) if x else "no file" for x in curr_dict["filename"]]
        caption = r"Multifunktionsgebäude -- sequenzielle Topologie- und Konfigurationsoptimierung\\Vergleich der Lebenszykluskosten für unterschiedliche maximale Kanalgeschwindigkeiten. Für die zulässigen Kanalgeschwindigkeiten von \SI{8}{\meter\per\second} bzw. \SIrange{10}{12}{\meter\per\second} lassen sich die Lebenszykluskosten im Vergleich zu \SI{5}{\meter\per\second} um \SI{5}{\percent} reduzieren -- die Reduktion ist auf reduzierte Kanalkosten zurückzuführen."
        save_data(fig, serializer, outname, filedata, caption)

#### maximale Variation

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        #"results/GPZ/combined/max_height_04m_no_velocity_limit/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_5_5ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_8_8ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_10_10ms_max_height_0.4m/VAV-VPC/",
        "results/GPZ/combined/duct_variations/max_velocity_12_12ms_max_height_0.4m/VAV-VPC/",
        ]

    # "FIXED FANS, VAR",

    names = ["5","8","10","12"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    


    for idx in range(len(curr_dict["mean_horizontal_velocity"])):
        t_bar = 0.2
        ax.bar(idx+t_bar, curr_dict["mean_vertical_velocity"][idx], color="k", width=t_bar*2)
        ax.bar(idx-t_bar, curr_dict["mean_horizontal_velocity"][idx], color="grey", width=t_bar*2)
        
    ax.set_xlabel("MITTLERE KANALGESCHWINDIGKEITEN in m/s")
    
    ax.legend(["Hauptkanal", "Abzweig"], prop=arial_font)

    ax.set_xticks(range(len(curr_dict["mean_horizontal_velocity"])), names)

    fig.tight_layout()

### Variable CAPEX

In [ ]:
standard_case_bar_chart = True

if standard_case_bar_chart:
    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/capex_reduction/"

    foldernames = ["0.8/ODS-CC/", "0.5/ODS-CC/", "0.1/ODS-CC/"]
    # foldernames = ["VAV-VPC"]
    filename_list = [folder + x for x in foldernames]

    filename_list.insert(0, "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/ODS-CC/")

    cost_reduction = [0, 1-0.8, 1-0.5, 1-0.1]
    # names = ["100","80", "50", "10"]

    standard_dict_list = create_standard_list(filename_list, "Configuration", use_connected_opt=True)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring


    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        # Extract relevant costs at arg_max_lcc index
        costs = [curr_dict[name][0]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(f"{cost_reduction[idx]*100:.0f}", cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(f"{cost_reduction[idx]*100:.0f}", cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    # ax.set_xticks(ticknames, ticknames, fontsize=7.5, multialignment="left")

    ax = add_legend_right(ax, entry_names)

    # ax.set_ylim(top=9e1)

    # ax.set_xticks( fontsize=5)

    # ax.set_xticks(np.arange(n_elements), [""]*n_elements)
    ax.set_xlabel("REDUKTION KAPITALBEZOGENER KOSTEN in %")

    fig.tight_layout()
    if save_plots:
        outname = "bar_chart_pareto_lcc_GPZ_capex_reduction"
        filedata = [x["filename"][0] for x in standard_dict_list]
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Vergleich der Lebenszykluskosten für reduzierte kapitalbezogene Kosten für Regelstrategie \gls{odscc}. Bei Reduktionen bis zu \SI{90}{\percent} ergibt sich dieselbe Regelstrategie (\gls{vavvpc}) als optimal, lediglich die Kanaldimensionen steigen leicht an, wodurch der Energieverbrauch um \SI{5}{\percent} sinkt."
        save_data(fig, serializer, outname, filedata, caption)

## Is holistic optimisation really necessary?
This is verified by comparing two optimisation runs. In the first run, the configuration (i.e., the decision regarding the control strategy and the purchased fans) is taken from the topology optimisation. In the second optimisation run, the configuration is left open.
If both optimisation results are identical, then holistic optimisation does not yield any benefit.

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath_fix = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    folderpath_var = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case_configopt_no_fixed_configuration/"

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    folderpath_fix = [folderpath_fix + cs for cs in foldernames]
    folderpath_var = [folderpath_var + cs for cs in foldernames]

    standard_dict_list_fix = create_standard_list(folderpath_fix, "Configuration", use_connected_opt=False)
    standard_dict_list_var = create_standard_list(folderpath_var, "Configuration", use_connected_opt=False)

    standard_dict_list_fix = merge_standard_list(standard_dict_list_fix)
    standard_dict_list_var = merge_standard_list(standard_dict_list_var)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator", "Energiekosten"]


    # Compute exact fan energy costs
    standard_dict_list_fix["exact_fan_energy_costs"] = (
        standard_dict_list_fix["fan_energy_costs"]
        * standard_dict_list_fix["exact_power_consumption"]
        / standard_dict_list_fix["power_consumption"]
    )
    # Compute exact fan energy costs
    standard_dict_list_var["exact_fan_energy_costs"] = (
        standard_dict_list_var["fan_energy_costs"]
        * standard_dict_list_var["exact_power_consumption"]
        / standard_dict_list_var["power_consumption"]
    )

        # Plot stacked bar
    for idx in range(len(standard_dict_list_fix["fan_energy_costs"])):
        costs_fix = [standard_dict_list_fix[name][idx]/1e3 for name in entry_names]
        costs_var = [standard_dict_list_var[name][idx]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost_fix, b_cost_var = 0, 0
        for i, (cost_fix, cost_var) in enumerate(zip(costs_fix, costs_var)):
            if i == len(costs_fix) - 1:
                ax.bar(idx-0.2, cost_fix, bottom=b_cost_fix, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3, width=.4)
                ax.bar(idx+0.2, cost_var, bottom=b_cost_var, color="white", hatch="xx", edgecolor="black", linewidth=0.3, width=.4)
            else:
                ax.bar(idx-0.2, cost_fix, bottom=b_cost_fix, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3, width=.4)
                ax.bar(idx+0.2, cost_var, bottom=b_cost_var, color=colors[i], edgecolor="black", linewidth=0.3, width=.4)
            b_cost_fix += cost_fix
            b_cost_var += cost_var

    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    ticknames = control_strategies_names_ger_twocolumn

    ax.set_xticks(range(len(standard_dict_list_fix["computation_time"])), ticknames, fontsize=9)

    ax = add_legend_right(ax, cost_label)
    fig.tight_layout()

## Error analysis

### Energetically

In [ ]:
optimality_gaps = True
if optimality_gaps:
    
    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    cs_names = foldernames

    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff',  '#dcdcdcff']

    # Compute indices of maximum LCC
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width*0.8,diss_tex_width*0.8*5/9))

    entry_names = ["primal solution MINLP", "primal solution MIP", "dual bound MIP"]

    filenames = []
    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        opt_idx = arg_max_lcc[idx]

        filenames.append(curr_dict["filename"][opt_idx])
        
        # Compute exact fan energy costs
        lcc_cost = curr_dict["fan_energy_costs"][opt_idx]/1e3 + curr_dict["invest_costs"][opt_idx]/1e3
        dual_bound_mip = (1-curr_dict["optimality_gap"][opt_idx]/100)*(lcc_cost)
        primal_bound_mip = lcc_cost
        primal_bound_minlp = curr_dict["exact_lcc"][opt_idx]/1e3

        mip_gap = lcc_cost - dual_bound_mip
        approx_error = primal_bound_minlp - lcc_cost

        ax.errorbar(control_strategies_names_ger_twocolumn[idx], primal_bound_minlp, yerr=[[mip_gap],[0]], fmt='o', markersize=3, capsize=5, color= "k")

        ax.errorbar(control_strategies_names_ger_twocolumn[idx], primal_bound_minlp, yerr=[[approx_error], [0]], fmt='o', markersize=3, capsize=5, color= "k")

    ax.set_ylabel("LEBENSZYKLUSKOSTEN\nin $10^3~$€")

    ax.errorbar(3.5, 85, yerr=[[4],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.errorbar(3.5, 85, yerr=[[8],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.text(3.7, 85, "primale Lösung MINLP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')
    ax.text(3.7, 81, "primale Lösung MILP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')
    ax.text(3.7, 77, "duale Lösung MILP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')
    
    ax.set_xticklabels(control_strategies_names_ger_twocolumn, fontsize=8)
    # ax.set_ylim(bottom=0)

    fig.tight_layout()       

In [ ]:
optimality_gaps = True
if optimality_gaps:
    
    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    cs_names = foldernames

    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff',  '#dcdcdcff']


    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width*0.8,diss_tex_width*0.8*5/9))

    entry_names = ["primal solution MINLP", "primal solution MIP", "dual bound MIP"]

    # Iterate over strategies
    curr_dict = standard_dict_list[-1]
    curr_dict["dual bound MIP"] = (1-curr_dict["optimality_gap"]/100)*(curr_dict["fan_energy_costs"]/1e3 + curr_dict["invest_costs"]/1e3)
    curr_dict["primal solution MIP"] = (curr_dict["fan_energy_costs"]/1e3 + curr_dict["invest_costs"]/1e3)
    curr_dict["primal solution MINLP"] = curr_dict["exact_lcc"]/1e3

    curr_dict["mip gap abs"] = curr_dict["primal solution MIP"] - curr_dict["dual bound MIP"]
    curr_dict["relax error abs"] = curr_dict["primal solution MINLP"] - curr_dict["primal solution MIP"]

    # Iterate over strategies
    # for idx, entry in enumerate(entry_names):
    first_err = curr_dict["relax error abs"]
    second_err = curr_dict["mip gap abs"] + curr_dict["relax error abs"]

    energy_costs = [f"{costs/1e3:.1f}" for costs in curr_dict["fan_energy_costs"]]

    ax.errorbar(energy_costs, curr_dict["primal solution MINLP"], yerr=[first_err, [0]*len(first_err)], fmt='o', markersize=3, capsize=5, color= "k")

    ax.errorbar(energy_costs, curr_dict["primal solution MINLP"], yerr=[second_err, [0]*len(first_err)], fmt='o', markersize=3, capsize=5, color= "k")

    ax.set_ylabel("LEBENSZYKLUSKOSTEN\nin $10^3~$€")

    ax.errorbar(3.5, 85, yerr=[[4],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.errorbar(3.5, 85, yerr=[[8],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.text(3.9, 85, "primale Lösung MINLP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')
    ax.text(3.9, 81, "primale Lösung MILP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')
    ax.text(3.9, 77, "duale Lösung MILP", va="center", fontsize=8, fontfamily='Arial', fontstyle='italic')

    # ax.set_xticklabels(range(len(first_err),0,-1), fontsize=8)
    # ax.set_ylim(bottom=0)
    # ax.set_xticks(range(len(first_err),0,-1), [""]*len(first_err))
    ax.set_xlabel("ENERGIEKOSTEN in $10^3~$€")

    ax.set_xticks(energy_costs, energy_costs,  fontsize=8)

    fig.tight_layout()

## Computation time

In [ ]:
optimality_gaps = True
if optimality_gaps:

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

    filename_list = [folder + x for x in foldernames]

    standard_dict_list1 = create_standard_list(filename_list, "Topology", use_connected_opt=False)
    standard_dict_list2 = create_standard_list(filename_list, "Configuration", use_connected_opt=True)

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width*0.8,diss_tex_width*0.8*5/9))

    # Iterate over strategies
    for idx, (curr_dict_topo, curr_dict_conf) in enumerate(zip(standard_dict_list1, standard_dict_list2)):
        if len(curr_dict_topo["computation_time"]) != 1 or len(curr_dict_conf["computation_time"]) != 1:
            raise ValueError("Too many results per control strategy")
        ax.bar(idx-0.15, curr_dict_topo["computation_time"][0], color= "grey", width=0.3)
        ax.bar(idx+0.15, curr_dict_conf["computation_time"][0], color= "black", width=0.3)

    ax.set_ylabel("RECHENZEIT in s")

    ax.set_xticks(np.arange(len(standard_dict_list1)), control_strategies_names_ger_twocolumn, fontsize=8)

    ax.legend(["Topologieoptimierung","Konfigurationsoptimierung"], prop=arial_font)

    fig.tight_layout()

    if save_plots:
        outname = "computation_time_GPZ_topo_config"
        filedata = [*[x["filename"][0] if x["filename"] else "no file" for x in standard_dict_list1], *[x["filename"][0] if x["filename"] else "no file" for x in standard_dict_list2]]
        caption = r"Multifunktionsgebäude\\Rechenzeiten der Regelstrategien für Topologie- und Konfigurationsoptimierung. "
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
optimality_gaps = True
if optimality_gaps:

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    cs_names = foldernames

    folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff',  '#dcdcdcff']

    # Compute indices of maximum LCC
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    # Iterate over strategies
    curr_dict = standard_dict_list[-1]

    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )

    curr_dict = sort_list_of_dicts([curr_dict], "exact_power_consumption")[0]

    ax.bar([f"{x:.0f}" for x in curr_dict["exact_power_consumption"]], curr_dict["computation_time"], color= "k")

    ax.set_ylabel("RECHENZEIT in s")
    ax.set_xticks([f"{x:.0f}" for x in curr_dict["exact_power_consumption"]], [f"{x:.0f}" for x in curr_dict["exact_power_consumption"]], fontsize=7)
    # ax.set_xticks([])
    ax.set_xlabel(r"MITTLERE LEISTUNG in W")
    ax.set_yscale("log")
    fig.tight_layout()

    if save_plots:
        outname = "computation_time_GPZ_topo_ODS_CC"
        filedata = [str(x) for x in curr_dict["filename"]]
        caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Rechenzeit für verschiedene Kompromisse zwischen den konfliktären Ziele Leistungsaufnahme und kapitalbezogene Kosten. Mit sinkender Leistungsaufnahme steigt die Rechenzeit deutlich an. Die beiden Rechnungen mit niedrigster Leistungsaufnahme wurden nach \SI{12}{\hour} mit endlicher Dualitätslücke abgebrochen."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
optimality_gaps = True
if optimality_gaps:
    folder = "results/GPZ/combined/real/rooms_identical_noise_limits_pareto/"

    foldernames = ["VAV-VPC", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Configuration", use_connected_opt=False)

    standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff',  '#dcdcdcff']

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    # Iterate over strategies
    curr_dict = standard_dict_list[-1]

    ax.bar([f"{costs:.1f}" for costs in curr_dict["max_noise_value"]], curr_dict["computation_time"], color= "k")

    ax.set_ylabel("RECHENZEIT in s")
    # ax.set_xticks(range(len(first_err),0,-1), [""]*len(first_err))
    # ax.set_xticks([])
    ax.set_xlabel("MAX. ZULÄSSIGER SCHALLDRUCKPEGEL in dB")
    ax.set_yscale("log")
    ax.set_yticks([10,100,1000])
    fig.tight_layout()

    if save_plots:
        outname = "computation_time_GPZ_config_noise_pareto_ODS_CC"
        filedata = [str(x) for x in curr_dict["filename"]]
        caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Rechenzeit für verschiedene Kompromisse der konfliktären Ziele Lebenszykluskosten und zulässigen Schalldruckpegeln. Mit sinkenden zulässigen Schalldruckpegeln, steigt die Rechenzeit auf ein Maximum von knapp \SI{10}{\min} an."
        save_data(fig, serializer, outname, filedata, caption)